# Dataset Preparation
**This notebook will not be shared with the students.**
Prepare data from the `hseb-benchmark/msmarco` HuggingFace dataset for the home assignment.

In [1]:
%cd ../

/Users/jakubzovakfilevine/Desktop/advanced_ai_driven_search


In [2]:
%load_ext autoreload
%autoreload 2

## Setup

In [3]:
from typing import cast
import random
from pathlib import Path

from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from fastembed import (
    LateInteractionTextEmbedding,
    SparseTextEmbedding,
    TextEmbedding,
)

## Load Models

In [ ]:
SPLADE_MODEL_NAME = "prithivida/Splade_PP_en_v1"
splade_model = SparseTextEmbedding(SPLADE_MODEL_NAME)

## Load Dataset
Load the existing HSBE dataset from HuggingFace.

In [ ]:
# Queries need to be created from running the retrieval pipeline
# query = load_dataset("hseb-benchmark/msmarco", "query-all-MiniLM-L6-v2-100K")

documents_dataset: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_semestral_assignment",
    name=None, 
    data_dir="corpus-all-MiniLM-L6-v2-50K-groups-multi-vector"
))

In [ ]:
columns_to_remove_from_documents = ["groups"]
documents_dataset = documents_dataset.remove_columns(columns_to_remove_from_documents)

## Save Modified Dataset Locally

## Load & Take Subset of Documents

In [5]:
# Number of documents to keep, expressed in thousands (e.g. 10 -> 10_000)
NUM_THOUSANDS = 1
new_size = NUM_THOUSANDS * 1_000
DOCUMENTS_DATASET_NAME = (
    f"corpus-all-MiniLM-L6-v2-{NUM_THOUSANDS}K-multi-vector-splade-metadata"
)

In [ ]:
# Create a subset of documents enriched with SPLADE vectors and rating metadata
random.seed(42)
documents_dataset_subset = DatasetDict()

for split_name in documents_dataset.keys():
    total_size = len(documents_dataset[split_name])

    subset = documents_dataset[split_name].select(range(new_size))

    # Precompute SPLADE sparse vectors so students don't have to run the model
    texts = subset["text"]
    splade_embeddings = list(splade_model.passage_embed(texts, batch_size=32))
    subset = subset.add_column(
        "splade_indices", [e.indices.tolist() for e in splade_embeddings]
    )
    subset = subset.add_column(
        "splade_values", [e.values.tolist() for e in splade_embeddings]
    )

    # Synthetic metadata: random integer rating from 1 to 5
    subset = subset.add_column(
        "rating", [random.randint(1, 5) for _ in range(new_size)]
    )

    documents_dataset_subset[split_name] = subset
    print(f"{split_name}: {total_size} -> {new_size} examples")

output_path = (Path("data") / DOCUMENTS_DATASET_NAME).resolve()
documents_dataset_subset.save_to_disk(str(output_path))
print(f"\nDataset saved to '{output_path}'")
# TODO: Upload the dataset to the Hugging Face dataset


## Queries Preprocessing

In [ ]:
query_dataset: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_semestral_assignment",
    data_dir="query-all-MiniLM-L6-v2-100-filters-embedded-results"
))

for split_name in query_dataset.keys():
    # Take first 100 queries
    query_dataset[split_name] = query_dataset[split_name].select(range(50))
    
    # Remove specified columns
    columns_to_remove = [
        'filters',
        'result'
    ]
    query_dataset[split_name] = query_dataset[split_name].remove_columns(columns_to_remove)

In [ ]:
query_dataset.save_to_disk("./data/query-all-MiniLM-L6-v2-20")

## Upload Final Datasets

In [ ]:
query_dataset_for_hub = DatasetDict.load_from_disk("./data/query-all-MiniLM-L6-v2-100-filters-embedded-results")

In [ ]:
query_dataset_for_hub["train"].to_json("./hub/data/query-all-MiniLM-L6-v2-100-filters-embedded-results/train.jsonl", orient="records", lines=True)

In [6]:
documents_dataset_for_hub = DatasetDict.load_from_disk(f"./data/{DOCUMENTS_DATASET_NAME}")

In [7]:
from pathlib import Path

# Convert the Arrow dataset (save_to_disk format) into a single train.jsonl,
# one JSON object per line, matching the layout of the previous Hub uploads.
documents_jsonl_path = Path("data") / "hub" / DOCUMENTS_DATASET_NAME / "train.jsonl"
documents_jsonl_path.parent.mkdir(parents=True, exist_ok=True)

documents_dataset_for_hub["train"].to_json(
    str(documents_jsonl_path), orient="records", lines=True
)
print(f"Wrote {documents_jsonl_path}")


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Wrote data/hub/corpus-all-MiniLM-L6-v2-1K-multi-vector-splade-metadata/train.jsonl


In [ ]:
from huggingface_hub import HfApi

# Upload the single train.jsonl into the repo under a folder named after the
# dataset, so it can be loaded back with data_dir=DOCUMENTS_DATASET_NAME.
HUB_REPO_ID = "Zovi3/pa195_semestral_assignment"

api = HfApi()
api.upload_file(
    path_or_fileobj=str(documents_jsonl_path),
    path_in_repo=f"{DOCUMENTS_DATASET_NAME}/train.jsonl",
    repo_id=HUB_REPO_ID,
    repo_type="dataset",
)
print(f"Uploaded {DOCUMENTS_DATASET_NAME}/train.jsonl to {HUB_REPO_ID}")

## Try to load

In [ ]:
query_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_test",
    name=None, 
    data_dir="query-all-MiniLM-L6-v2-100-filters-embedded-results"
)) 
query_dataset: Dataset = query_dataset_dict["train"]
query_dataset

In [ ]:
documents_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_test",
    name=None, 
    data_dir="corpus-all-MiniLM-L6-v2-50K-groups-multi-vector"
)) 
documents: Dataset = query_dataset_dict["train"]
documents